# 07 — Testing zone: how much does each step improve the recommendations?

**Purpose.** Each step so far was tested on its own (do the tiers separate good from bad movies, do the vibe scores match the Genome). This notebook tests the **recommendations themselves**: one fixed benchmark that every version of the recommender runs through the same way, with results appended to a **scoreboard** (`artifacts/scoreboard.csv`). The scoreboard shows what each step gains or costs, in accuracy and in speed. Roadmap Step 12 (evaluation + ablation) grows out of it.

**Benchmark (PROJECT.md §12: time-based split)**
- **Cutoff:** 2018-01-04, the 90th percentile of all rating timestamps, the same cutoff as Step 4's validation. Ratings before it are what the recommender may see; ratings after it are "the future".
- **Test users:** a random sample of 5,000 users (seed 42) with at least one rating before the cutoff and at least one hit after it. The sample size is a computing setting, not a result threshold.
- **Hit:** a movie the user rated after the cutoff **above their own pre-cutoff average** (Adil's choice: relative to each user, so harsh and generous raters are treated alike).

**No peeking at the future.** Popularity, fair scores and tiers are all recomputed from pre-cutoff ratings, with the same Step 4 rule.
- Vibe scores come from Wikipedia/TMDB text and the Tag Genome. The Genome was built from tags collected up to 2019, a small leak, noted as a limitation.
- Movies with no rating before the cutoff cannot be recommended by any method here; future hits on them count as misses for everyone alike.

**Metrics**

| Kind | Metric | Meaning |
|---|---|---|
| Accuracy | recall@10, recall@50 | share of the user's future hits found in the top 10 / 50 |
| | ndcg@10 | like recall, but a hit at #1 counts more than at #10 |
| Speed | ms_per_user, candidates_scanned | time and number of movies examined per user |
| Beyond accuracy | coverage@10 | share of the pool that appears in anyone's top 10 |
| | diversity@10 | how different the 10 picks are from each other (vibe distance) |
| | novelty@10 | how un-obvious the picks are (−log2 of their share of ratings) |

Code: `src/viberec/evaluation.py` (benchmark, metrics, scoreboard), `src/viberec/recommenders.py` (the recommenders).

## 1. Config and load

In [ ]:
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))
from src.viberec import credibility as cr, evaluation as ev, recommenders as rc, tiers as tr, genome as gn

PROCESSED_DIR = Path("..") / "data" / "processed"
ARTIFACTS_DIR = Path("..") / "artifacts"
RATINGS_CSV = Path("..") / "data" / "raw" / "movielens" / "ml-25m" / "ratings.csv"
SCOREBOARD = ARTIFACTS_DIR / "scoreboard.csv"
VERSION = "v1"
N_USERS, SEED, K_SHRINK = 5000, 42, 7.324        # sample size (computing setting); k from Q4.1 / Q4.3


def write_metadata(path, **info):
    meta = {"created_at": datetime.now(timezone.utc).isoformat(), "produced_by": "notebooks/07_testing_zone.ipynb", **info}
    path.with_name(path.stem + ".metadata.json").write_text(json.dumps(meta, indent=2, default=str))


early = pd.read_parquet(ARTIFACTS_DIR / f"q43_early_counts_{VERSION}.parquet")
early = early[early["quantile"] == 0.9]                                     # the 2018-01-04 cutoff
CUTOFF_TS = int(early["cutoff_ts"].iloc[0])
print("cutoff:", pd.to_datetime(CUTOFF_TS, unit="s").date())

## 2. The benchmark: test users and their future hits

Built once from `ratings.csv` (25M rows, about a minute to read) and saved: the test users' pre-cutoff ratings and their future hits. Later runs load the saved files.

In [ ]:
TRAIN_PATH = PROCESSED_DIR / f"benchmark_train_{VERSION}.parquet"
HITS_PATH = PROCESSED_DIR / f"benchmark_hits_{VERSION}.parquet"
if TRAIN_PATH.exists() and HITS_PATH.exists():
    print("benchmark already exists, loading.")
    train = pd.read_parquet(TRAIN_PATH)
    hits_df = pd.read_parquet(HITS_PATH)
    meta = json.loads(TRAIN_PATH.with_name(TRAIN_PATH.stem + ".metadata.json").read_text())
    eligible = meta["eligible_users"]
else:
    start = time.time()
    ratings = pd.read_csv(RATINGS_CSV, dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"})
    train, hits, _, eligible = ev.make_benchmark(ratings, CUTOFF_TS, n_users=N_USERS, seed=SEED)
    hits_df = pd.DataFrame([(u, m) for u, ms in hits.items() for m in ms], columns=["userId", "movieId"])
    train.to_parquet(TRAIN_PATH, index=False)
    hits_df.to_parquet(HITS_PATH, index=False)
    for p in (TRAIN_PATH, HITS_PATH):
        write_metadata(p, cutoff=str(pd.to_datetime(CUTOFF_TS, unit="s")), users=N_USERS, seed=SEED, eligible_users=eligible,
                       hit_rule="rated after the cutoff above the user's own pre-cutoff mean",
                       runtime_seconds=round(time.time() - start, 1))
    del ratings
hits = hits_df.groupby("userId")["movieId"].apply(set).to_dict()
user_mean = train.groupby("userId")["rating"].mean()
print(f"{len(hits):,} test users (of {eligible:,} eligible); pre-cutoff ratings per user: median "
      f"{train.groupby('userId').size().median():.0f}; future hits per user: median {np.median([len(h) for h in hits.values()]):.0f}")

## 3. What was known at the cutoff: popularity, fair score, tiers

- **Popularity:** each movie's number of pre-cutoff ratings (all users).
- **Fair score:** the Step 4 shrunk mean (k = 7.3, prior = mean of per-movie means), from pre-cutoff ratings.
- **Tiers:** the Step 4 rule (too low on the raw mean, too high on the shrunk score), run on pre-cutoff data with 2018 as "now", exactly as in Step 4's validation.

These give three candidate pools: every movie rated before the cutoff, tiers 2 + 3, and tier 3.

In [ ]:
movies = pd.read_parquet(PROCESSED_DIR / "movies_full.parquet")
movies["genres"] = movies["genres"].str.replace(",", "|", regex=False)
known = ev.movies_known_at(early, movies, K_SHRINK, cutoff_year=pd.to_datetime(CUTOFF_TS, unit="s").year)

popularity = known.set_index("movieId")["rating_count"]
popularity_share = (popularity / popularity.sum()).to_dict()
fair_score = known.set_index("movieId")["shrunk"]
pools = {"all_rated": known["movieId"].to_numpy(),
         "tier_2_3": known.loc[known["tier"].isin([2, 3]), "movieId"].to_numpy(),
         "tier_3": known.loc[known["tier"] == 3, "movieId"].to_numpy()}
print({name: len(p) for name, p in pools.items()}, "candidate movies at the cutoff")

# saved so other notebooks (e.g. Step 6) score against exactly the same cutoff-time facts
AT_CUTOFF_PATH = PROCESSED_DIR / f"benchmark_movies_at_cutoff_{VERSION}.parquet"
if not AT_CUTOFF_PATH.exists():
    known[["movieId", "rating_count", "mean_rating", "shrunk", "tier"]].to_parquet(AT_CUTOFF_PATH, index=False)
    write_metadata(AT_CUTOFF_PATH, cutoff="2018-01-04", note="per movie, from pre-cutoff ratings only: count, mean, fair score, Step 4 tier")

## 4. Baselines on the scoreboard

| Method | What it knows | Why it is here |
|---|---|---|
| **popular** | nothing about the user: the most-rated movies they have not seen | the classic, surprisingly strong baseline; anything personal must beat it |
| **best_rated** | nothing about the user: the highest fair scores | does quality alone help? |
| **vibe_match** | the user's taste in vibe (mean vibe of the movies they rated above their average) | the value of Step 5 in recommendations |

**Pools.** `popular` is run on all three pools: that measures Step 4's claim that dropping tier 1 makes the work smaller without losing hits. `vibe_match` runs on tier 3 movies that have vibe scores.

**Vibe scale.** Vibe vectors are standardised first (each vibe as "how far from the average movie"), so vibes that are high for every movie do not dominate.

In [ ]:
vibe_scores = pd.read_parquet(PROCESSED_DIR / "vibe_scores_v2.parquet").set_index("movieId")
VIBES = [c for c in vibe_scores.columns if c != "source"]
vibes = vibe_scores[VIBES].astype(float)
vibe_vectors = {m: v for m, v in zip(vibes.index, rc.standardise(vibes).to_numpy())}   # for diversity@10

runs = {
    ("4 quality tiers", "popular / all rated"): (rc.make_ranked_list(pools["all_rated"], popularity), len(pools["all_rated"])),
    ("4 quality tiers", "popular / tier 2+3"): (rc.make_ranked_list(pools["tier_2_3"], popularity), len(pools["tier_2_3"])),
    ("4 quality tiers", "popular / tier 3"): (rc.make_ranked_list(pools["tier_3"], popularity), len(pools["tier_3"])),
    ("4 quality tiers", "best_rated / tier 3"): (rc.make_ranked_list(pools["tier_3"], fair_score), len(pools["tier_3"])),
}
vibe_pool = np.intersect1d(pools["tier_3"], vibes.index)
runs[("5 vibe features", "vibe_match / tier 3 with vibes")] = (rc.make_vibe_match(vibes, vibe_pool, user_mean), len(vibe_pool))

results, per_user = {}, {}
for (step, method), (recommend, pool_size) in runs.items():
    start = time.time()
    results[method], per_user[method] = ev.evaluate(recommend, train, hits, vibe_vectors=vibe_vectors,
                                                    popularity_share=popularity_share, pool_size=pool_size,
                                                    return_per_user=True)
    ev.append_scoreboard(SCOREBOARD, step, method, results[method],
                         settings={"cutoff": "2018-01-04", "users": len(hits), "seed": SEED, "pool_size": pool_size},
                         note="baseline")
    print(f"{method:32s} recall@10 {results[method]['recall@10']:.4f}  recall@50 {results[method]['recall@50']:.4f}  "
          f"ndcg@10 {results[method]['ndcg@10']:.4f}  {results[method]['ms_per_user']:.2f} ms/user  ({time.time() - start:.0f}s)")

### Are the differences real? (paired, user by user)

Each test user is scored by both methods, so the fair comparison is per user: the difference for each user, averaged, with a bootstrap 95% interval (resampling users). An interval that excludes 0 means a real difference.

In [ ]:
comparisons = {"vibe_match vs popular (tier 3)": ("vibe_match / tier 3 with vibes", "popular / tier 3"),
               "best_rated vs popular (tier 3)": ("best_rated / tier 3", "popular / tier 3"),
               "popular: tier 3 vs all rated": ("popular / tier 3", "popular / all rated")}
for label, (a, b) in comparisons.items():
    parts = []
    for metric in ("recall@10", "recall@50", "ndcg@10"):
        d, lo, hi, rel = ev.paired_gain(per_user[a], per_user[b], metric, seed=SEED)
        parts.append(f"{metric} {d:+.4f} [{lo:+.4f}, {hi:+.4f}] ({rel:+.0%})")
    print(f"{label}:\n   " + "\n   ".join(parts))

## 5. The scoreboard so far

In [ ]:
board = pd.read_csv(SCOREBOARD)
cols = ["date", "step", "method", "recall@10", "recall@50", "ndcg@10", "coverage@10", "diversity@10",
        "novelty@10", "ms_per_user", "candidates_scanned"]
print(board[cols].round(4).to_string(index=False))

**Results (5,000 test users, cutoff 2018-01-04)**

| Method | recall@10 | recall@50 | ndcg@10 | coverage@10 | novelty@10 | diversity@10 | ms/user | candidates |
|---|---|---|---|---|---|---|---|---|
| popular / all rated | 0.0229 | 0.0682 | 0.0696 | 0.7% | 8.8 | 0.55 | 0.03 | 38,957 |
| popular / tier 3 | 0.0229 | 0.0682 | 0.0696 | 0.9% | 8.8 | 0.55 | 0.03 | 30,063 |
| best_rated / tier 3 | 0.0170 | 0.0486 | 0.0510 | 0.4% | 10.4 | 0.38 | 0.03 | 30,063 |
| **vibe_match / tier 3** | **0.0290** | **0.0913** | 0.0669 | **5.8%** | **17.0** | 0.25 | 0.91 | 27,752 |

How to read the numbers: recall@10 = 0.029 means about 3% of a user's future favourites appear in their top 10. Absolute numbers are low on any MovieLens time split (users rate many movies and only a few land in a top 10); what matters here is the **relative** change between methods.

- **Step 4 (tiers): smaller pool, nothing lost.** Dropping tiers 1 and 2 shrinks the candidate pool by 23% (38,957 → 30,063) with identical accuracy, because no future favourite fell in the dropped tiers for this method. For a fixed ranked list the time saving is negligible; it will matter for per-user scoring (vibe matching, CF) where the cost grows with the pool.
- **Quality alone (best_rated) is worse than popularity.**
- **Step 5 (vibes) adds real personal value:** vibe_match finds more future favourites than popularity (recall@10 and recall@50 up; see the paired intervals above). It also spreads recommendations over 6× more of the catalog and picks far less obvious movies (novelty 17 vs 9).
- **Paired check:** recall@10 +27% (95% interval +0.0030 to +0.0092), recall@50 +34% (+0.0178 to +0.0285); ndcg@10 −4% with an interval that includes 0, so no real difference at the very top. best_rated is clearly worse than popular (−26% recall@10).
- **Its costs:**
  - lower diversity (it recommends movies that feel alike);
  - 30× slower per user, though still under 1 ms.

  These are what later steps (CF, ranker, clustering) should improve.

**Limitations.**
- The Tag Genome was built from tags up to 2019 (slight future leak for Genome-based vibe scores).
- Movies first rated after the cutoff cannot be recommended by any method.
- 5,000 users is a sample.